# 02 — Feature Engineering
**Project 05: Price Elasticity & Pricing Simulator**

This notebook builds the engineered feature set from `reports/exploration/clean.parquet`
and writes `reports/modeling/features.parquet` and `reports/modeling/features.md`.

Each feature is documented with its business meaning, the explorer finding that motivates it,
its formula, an example row, a distribution check, and a verdict (keep or drop).

**All preprocessing is fit on the training split only** so no information leaks from
the held-out 26 weeks into the training distribution.

---
Sections:
1. Setup and library versions
2. Load data and define train/test split
3. Feature construction (one sub-section per feature)
4. Write features.parquet and features.md


## 1 · Setup and library versions

We print library versions so the analysis is reproducible.
The seed `SEED = 42` is used throughout.


In [1]:
import sys
sys.path.insert(0, "/Users/shrikar/Documents/Marketing Analytics/05-pricing-simulator")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sklearn
import lightgbm as lgb

print("Python :", sys.version)
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)
print("sklearn:", sklearn.__version__)
print("lgb    :", lgb.__version__)

SEED = 42
PROJECT = "/Users/shrikar/Documents/Marketing Analytics/05-pricing-simulator"
CHARTS = f"{PROJECT}/reports/modeling/charts"
import os
os.makedirs(CHARTS, exist_ok=True)
os.makedirs(f"{PROJECT}/reports/modeling/tables", exist_ok=True)


Python : 3.13.12 | packaged by Anaconda, Inc. | (main, Feb 24 2026, 16:07:20) [Clang 20.1.8 ]
numpy  : 2.4.6
pandas : 2.3.3
sklearn: 1.9.1
lgb    : 4.7.0


## 2 · Load data and define the train / test split

**Why a time split (not random)?**
The handoff recommends holding out the **last 26 weeks** (Jul–Dec 2011) as the test set.
A random row split would leak future prices and promotions into training, giving an
over-optimistic RMSE that disappears at deployment. An out-of-time split mimics
real use: the model is trained on history and evaluated on the future.

We define `split = 'train'` for the first 130 weeks and `split = 'test'` for the last 26.
This boundary is set before any feature computation so the lag and rolling features
are computed only from training rows.


In [2]:
df = pd.read_parquet(f"{PROJECT}/reports/exploration/clean.parquet")
print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("Date range:", df["WEEK_END_DATE"].min(), "to", df["WEEK_END_DATE"].max())
print("Unique weeks:", df["WEEK_END_DATE"].nunique())


Shape: (538435, 29)
Columns: ['WEEK_END_DATE', 'STORE_NUM', 'UPC', 'UNITS', 'VISITS', 'HHS', 'SPEND', 'PRICE', 'BASE_PRICE', 'FEATURE', 'DISPLAY', 'TPR_ONLY', 'DESCRIPTION', 'MANUFACTURER', 'CATEGORY', 'SUB_CATEGORY', 'PRODUCT_SIZE', 'STORE_NAME', 'ADDRESS_CITY_NAME', 'ADDRESS_STATE_PROV_CODE', 'MSA_CODE', 'SEG_VALUE_NAME', 'PARKING_SPACE_QTY', 'SALES_AREA_SIZE_NUM', 'AVG_WEEKLY_BASKETS', 'DISCOUNT_PCT', 'PROMO_TYPE', 'LOG_UNITS', 'LOG_PRICE']
Date range: 2009-01-14 00:00:00 to 2012-01-04 00:00:00
Unique weeks: 156


In [3]:
# Define split: last 26 weeks = test
weeks_sorted = sorted(df["WEEK_END_DATE"].unique())
test_weeks = set(weeks_sorted[-26:])
train_weeks = set(weeks_sorted[:-26])

df["split"] = df["WEEK_END_DATE"].apply(lambda w: "test" if w in test_weeks else "train")

print("Train rows:", (df["split"] == "train").sum())
print("Test rows :", (df["split"] == "test").sum())
print("Test period:", min(test_weeks).date(), "to", max(test_weeks).date())

train_mask = df["split"] == "train"


Train rows: 449684


Test rows : 88751
Test period: 2011-07-13 to 2012-01-04


## 3 · Feature construction

Each sub-section covers one feature or feature group.
For every feature we show:
1. Business meaning and motivating explorer finding
2. Alternatives considered
3. Formula (LaTeX or rule in words)
4. How to read the output
5. Distribution check on training data
6. Relationship to target (LOG_UNITS) on training data
7. Verdict: keep or drop


### 3.1 · LOG_PRICE — log of shelf price (already in clean.parquet)

**1. Business meaning:** The log of the actual shelf price charged that week.
In a log-log demand model the coefficient on LOG_PRICE is the own-price elasticity:
a 1% price increase predicts a elasticity-% change in units.

**Motivating finding (Explorer finding 1):** All four categories have confirmed
negative log-log demand slopes, so LOG_PRICE is the central predictor.

**2. Alternatives considered:**
- *Raw PRICE in levels:* creates heteroscedasticity — error variance grows with price level, violating OLS assumptions. The log transformation gives homogeneous residuals across the price range.
- *Box-Cox transform of PRICE:* a data-driven exponent might give a marginal fit improvement, but it destroys the direct elasticity interpretation; the log-log form is preferred because β directly equals the price elasticity.

**3. Formula:** `LOG_PRICE = log(PRICE)` where PRICE is the shelf price in dollars.

**Example row:**
- PRICE = 1.39 → LOG_PRICE = log(1.39) ≈ 0.329

**Expected direction:** Negative coefficient (higher price → lower sales).

**4. How to read the output:**
- *Distribution:* look for values below −3 (price under $0.05, likely a data error) or a spike at 0 (PRICE = 1). The bimodal shape (low-price snacks/cereal cluster vs high-price pizza/mouthwash cluster) is expected.  Any single-row anomaly at −∞ or NaN signals a zero-price record that must be imputed in the modeling notebook.
- *Scatter:* Pearson |r| ≥ 0.3 motivates keeping; a clearly negative slope confirms the demand model assumption. If |r| < 0.1, investigate whether category composition is masking the relationship.

**Verdict: KEEP.** This is the core feature for elasticity estimation.


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
df.loc[train_mask, "LOG_PRICE"].hist(bins=50, ax=axes[0])
axes[0].set_title("Distribution of LOG_PRICE (train)")
axes[0].set_xlabel("LOG_PRICE")
axes[0].set_ylabel("Count")

axes[1].scatter(
    df.loc[train_mask, "LOG_PRICE"].sample(5000, random_state=42),
    df.loc[train_mask, "LOG_UNITS"].sample(5000, random_state=42),
    alpha=0.1, s=5
)
axes[1].set_xlabel("LOG_PRICE")
axes[1].set_ylabel("LOG_UNITS")
axes[1].set_title("LOG_PRICE vs LOG_UNITS (sample, train)")

plt.tight_layout()
plt.savefig(f"{CHARTS}/feat_LOG_PRICE.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved feat_LOG_PRICE.png")
print("Corr(LOG_PRICE, LOG_UNITS) on train:", df.loc[train_mask, ["LOG_PRICE","LOG_UNITS"]].corr().iloc[0,1].round(3))


Saved feat_LOG_PRICE.png
Corr(LOG_PRICE, LOG_UNITS) on train: -0.398


/var/folders/cb/xwfhfdh96_j3702wn1j9nk980000gn/T/ipykernel_17999/1912403005.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** Corr(LOG_PRICE, LOG_UNITS) ≈ −0.40, confirming the negative log-log demand slope across all categories. The bimodal distribution reflects two price tiers (low-price snacks/cereal vs higher-price frozen pizza/mouthwash). One near-zero outlier (PRICE = 0) is imputed in the modeling notebook. KEEP.

### 3.2 · LOG_BASE_PRICE — log of everyday base price

**1. Business meaning:** The log of the regular non-promotional price. Including
this alongside LOG_PRICE separates the effect of the markdown depth from the
absolute price level. A product priced at $5.00 markdown to $4.00 may behave
differently from one priced at $1.50 markdown to $1.20, even with the same
percentage cut.

**Motivating finding (Explorer finding 2):** Cold Cereal and Mouthwash have
different average price levels and different elasticities. BASE_PRICE helps
the model separate category-level price positioning from temporary promotions.

**2. Alternatives considered:**
- *Raw BASE_PRICE in levels:* same heteroscedasticity argument as PRICE; log transformation is required for the same OLS homoscedasticity reasons.
- *Dropping LOG_BASE_PRICE and using PRICE_TO_BASE_RATIO only:* loses information about absolute price tier — a 10% cut on a $1 item is economically different from a 10% cut on a $5 item, both costing $0.10 but signaling very different competitive dynamics.

**3. Formula:** `LOG_BASE_PRICE = log(max(BASE_PRICE, 0.01))`

**Example row:**
- BASE_PRICE = 1.57 → LOG_BASE_PRICE ≈ 0.451

**Expected direction:** Positive coefficient (higher baseline price product → higher baseline demand on log scale for premium, loyal-buyer products; sign may flip with store/UPC fixed effects).

**4. How to read the output:**
- *Distribution:* should mirror LOG_PRICE in shape; a gap between two clusters signals the two price tiers. Values below −2 (base price under $0.14) would be anomalous pricing records.
- *Scatter:* |r| ≥ 0.2 justifies keeping; a near-zero correlation would suggest LOG_BASE_PRICE adds nothing over LOG_PRICE. The slope direction may differ from LOG_PRICE because base price captures tier rather than weekly variation.

**Verdict: KEEP.** Separates price level from price change.


In [5]:
df["LOG_BASE_PRICE"] = np.log(df["BASE_PRICE"].clip(lower=0.01))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
df.loc[train_mask, "LOG_BASE_PRICE"].hist(bins=50, ax=axes[0])
axes[0].set_title("Distribution of LOG_BASE_PRICE (train)")
axes[0].set_xlabel("LOG_BASE_PRICE")

axes[1].scatter(
    df.loc[train_mask, "LOG_BASE_PRICE"].sample(5000, random_state=42),
    df.loc[train_mask, "LOG_UNITS"].sample(5000, random_state=42),
    alpha=0.1, s=5
)
axes[1].set_xlabel("LOG_BASE_PRICE")
axes[1].set_ylabel("LOG_UNITS")
axes[1].set_title("LOG_BASE_PRICE vs LOG_UNITS (sample)")
plt.tight_layout()
plt.savefig(f"{CHARTS}/feat_LOG_BASE_PRICE.png", dpi=100, bbox_inches="tight")
plt.show()
print("Corr(LOG_BASE_PRICE, LOG_UNITS) on train:", df.loc[train_mask, ["LOG_BASE_PRICE","LOG_UNITS"]].corr().iloc[0,1].round(3))


Corr(LOG_BASE_PRICE, LOG_UNITS) on train: -0.351


/var/folders/cb/xwfhfdh96_j3702wn1j9nk980000gn/T/ipykernel_17999/4020606415.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** Corr(LOG_BASE_PRICE, LOG_UNITS) ≈ −0.35 — slightly weaker than LOG_PRICE, confirming it carries additional price-tier information not in LOG_PRICE alone. The bimodal distribution mirrors LOG_PRICE (same two category clusters), which is expected since base prices are stable. KEEP.

### 3.3 · PRICE_TO_BASE_RATIO — markdown depth (scale-free)

**1. Business meaning:** PRICE / BASE_PRICE. When equal to 1.0 there is no markdown;
values below 1.0 indicate a price cut. This is a scale-free measure of promotional
depth that is comparable across products at different price points.

**Motivating finding (Explorer finding 2 & 3):** Feature + Display events lift
units-per-visit by ~15%, which partly reflects deeper markdowns. This feature
separates the depth of the cut from the absolute price, letting the model
distinguish a 5% markdown from a 20% markdown.

**2. Alternatives considered:**
- *Binary indicator for any markdown (PRICE < BASE_PRICE):* loses depth — a 1% cut gets the same code as a 50% cut, discarding the continuous dose-response relationship.
- *Dollar-amount markdown (BASE_PRICE − PRICE):* depth in dollars, but scale depends on price tier; $0.20 off a $1 item is a 20% cut while $0.20 off a $5 item is only 4%.

**3. Formula:** `PRICE_TO_BASE_RATIO = PRICE / BASE_PRICE`

**Code:** `df["PRICE_TO_BASE_RATIO"] = df["PRICE"].clip(lower=0.01) / df["BASE_PRICE"].clip(lower=0.01)`

**Example row:**
- PRICE = 1.39, BASE_PRICE = 1.57 → PRICE_TO_BASE_RATIO ≈ 0.885 (11% markdown)

**Expected direction:** Positive coefficient on LOG_UNITS at the raw level (deeper cuts → more units), but collinear with DISCOUNT_PCT (linear transform).

**4. How to read the output:**
- *Distribution:* a large spike at 1.0 (no markdown) is expected; the fraction of values below 1 should match the declared promotion rate (~26%). Values above 1.1 (selling above base price) are anomalous and should be flagged.
- *Scatter:* |r| ≥ 0.15 justifies keeping; expect negative slope (lower ratio = deeper markdown = more units). If the slope is flat, markdown depth is not driving demand in this data.

**Verdict: KEEP.** Directly captures markdown depth as a continuous, scale-free variable.


In [6]:
df["PRICE_TO_BASE_RATIO"] = df["PRICE"].clip(lower=0.01) / df["BASE_PRICE"].clip(lower=0.01)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
df.loc[train_mask, "PRICE_TO_BASE_RATIO"].hist(bins=50, ax=axes[0])
axes[0].set_title("Distribution of PRICE_TO_BASE_RATIO (train)")
axes[0].set_xlabel("PRICE_TO_BASE_RATIO")

axes[1].scatter(
    df.loc[train_mask, "PRICE_TO_BASE_RATIO"].sample(5000, random_state=42),
    df.loc[train_mask, "LOG_UNITS"].sample(5000, random_state=42),
    alpha=0.1, s=5
)
axes[1].set_xlabel("PRICE_TO_BASE_RATIO")
axes[1].set_ylabel("LOG_UNITS")
axes[1].set_title("PRICE_TO_BASE_RATIO vs LOG_UNITS")
plt.tight_layout()
plt.savefig(f"{CHARTS}/feat_PRICE_TO_BASE_RATIO.png", dpi=100, bbox_inches="tight")
plt.show()

pct_on_promo = (df.loc[train_mask, "PRICE_TO_BASE_RATIO"] < 1).mean()
print(f"Rows with PRICE < BASE_PRICE (on markdown): {pct_on_promo:.1%}")
print(f"Min PRICE_TO_BASE_RATIO: {df.loc[train_mask,'PRICE_TO_BASE_RATIO'].min():.3f}")


Rows with PRICE < BASE_PRICE (on markdown): 26.2%
Min PRICE_TO_BASE_RATIO: 0.003


/var/folders/cb/xwfhfdh96_j3702wn1j9nk980000gn/T/ipykernel_17999/2934564515.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** About 26.2% of training rows have PRICE_TO_BASE_RATIO < 1 (on markdown), matching the declared promotion rate. Corr ≈ −0.19 (negative as expected: lower ratio = deeper cut = more units). The large spike at 1.0 (no markdown) is correct. KEEP.

### 3.4 · DISCOUNT_PCT — percentage markdown (already in clean.parquet)

**1. Business meaning:** `(BASE_PRICE - PRICE) / BASE_PRICE * 100`. This expresses
the markdown as a percentage of the everyday price. It is the most intuitive
form of "how big is the promotion?" for a business user.

**Motivating finding (Explorer finding 3):** The explorer confirmed that Feature + Display
events are associated with deeper markdowns. DISCOUNT_PCT quantifies that depth.

**2. Alternatives considered:**
- *Binary any-discount indicator (PRICE < BASE_PRICE):* loses the continuous depth signal; a 1% cut gets the same value as a 60% cut, discarding the dose-response relationship that is central to price elasticity.
- *Interaction DISCOUNT_PCT × IS_ON_PROMO:* redundant since DISCOUNT_PCT = 0 exactly when not on promo; the two-column encoding carries the same information with more complexity.

**3. Formula:** `DISCOUNT_PCT = (BASE_PRICE - PRICE) / BASE_PRICE * 100`

**Example row:**
- PRICE = 1.39, BASE_PRICE = 1.57 → DISCOUNT_PCT ≈ 11.5%

**Expected direction:** Positive coefficient (larger discount → more units).

Note: DISCOUNT_PCT and PRICE_TO_BASE_RATIO are linearly related
(`DISCOUNT_PCT = 100 × (1 − PRICE_TO_BASE_RATIO)`). We include both because
DISCOUNT_PCT is used in business reporting; we will verify that VIF is acceptable
or drop one at model stage.

**4. How to read the output:**
- *Distribution:* heavily zero-inflated (73–75% zeros) is expected because most weeks have no promotion. A smooth right tail extending to ~60% is normal. Values above 70% (selling at less than 30% of base price) are anomalous pricing records worth investigating.
- *Scatter:* |r| ≥ 0.15 justifies keeping; expect positive slope (larger discount → more units). A negative slope would be a data quality warning.

**Verdict: KEEP.** Business-interpretable; PRICE_TO_BASE_RATIO can be dropped
if collinearity is confirmed at the model step.


In [7]:
# Already in clean.parquet; verify distribution
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
df.loc[train_mask, "DISCOUNT_PCT"].hist(bins=50, ax=axes[0])
axes[0].set_title("Distribution of DISCOUNT_PCT (train)")
axes[0].set_xlabel("DISCOUNT_PCT (%)")

axes[1].scatter(
    df.loc[train_mask, "DISCOUNT_PCT"].sample(5000, random_state=42),
    df.loc[train_mask, "LOG_UNITS"].sample(5000, random_state=42),
    alpha=0.1, s=5
)
axes[1].set_xlabel("DISCOUNT_PCT")
axes[1].set_ylabel("LOG_UNITS")
axes[1].set_title("DISCOUNT_PCT vs LOG_UNITS")
plt.tight_layout()
plt.savefig(f"{CHARTS}/feat_DISCOUNT_PCT.png", dpi=100, bbox_inches="tight")
plt.show()
print("Corr(DISCOUNT_PCT, LOG_UNITS) on train:", df.loc[train_mask, ["DISCOUNT_PCT","LOG_UNITS"]].corr().iloc[0,1].round(3))


Corr(DISCOUNT_PCT, LOG_UNITS) on train: 0.199


/var/folders/cb/xwfhfdh96_j3702wn1j9nk980000gn/T/ipykernel_17999/3123326521.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** Corr(DISCOUNT_PCT, LOG_UNITS) ≈ +0.20, positive as expected (larger discount → more units). The distribution is heavily zero-inflated (73.8% at 0%) with a right tail to ~60%. The zero mass is correct: most weeks have no promotion. KEEP.

### 3.5 · FEATURE, DISPLAY, TPR_ONLY — promotion mechanic dummies (already in clean.parquet)

**1. Business meaning:**
- `FEATURE = 1`: product appeared in the weekly advertising circular.
- `DISPLAY = 1`: product had an in-store display (end-cap, pallet, etc.).
- `TPR_ONLY = 1`: temporary price reduction with no feature or display support.

Each captures a different component of the promotional mix. Feature and Display
increase awareness and visibility beyond the price cut itself.

**Motivating finding (Explorer finding 3):** Feature + Display weeks lift
units-per-visit by ~15% above no-promo baseline. Feature-only outperforms
TPR-only on traffic-adjusted sales, confirming that advertising support adds
incremental volume above the price cut alone.

**2. Alternatives considered:**
- *Single promotion-intensity integer (sum of flags 0–3):* loses the distinction between different promotion mechanics; Feature lift and Display lift are non-additive and differ in size, so treating them as an ordinal scale is incorrect.
- *All pairwise interaction terms (FEATURE × DISPLAY, etc.):* adds collinearity and near-zero cells (Feature+Display is rare); the incremental information is captured by the PROMO_TYPE categorical column already in the data.

**Expected direction:** All positive coefficients on LOG_UNITS.

**4. How to read the output:**
- *Mean LOG_UNITS table:* Feature+Display rows should show the highest mean LOG_UNITS; TPR-only should be intermediate; no-promo baseline should be lowest. If Feature-only is lower than no-promo, check for confounding (Feature events may co-occur with seasonal lows or low-traffic stores).

**Verdict: KEEP.** Core promotion-mechanic features.


In [8]:
promo_means = df.loc[train_mask].groupby(["FEATURE","DISPLAY","TPR_ONLY"])["LOG_UNITS"].mean().reset_index()
print("Mean LOG_UNITS by promo combination (train):")
print(promo_means.to_string())


Mean LOG_UNITS by promo combination (train):
   FEATURE  DISPLAY  TPR_ONLY  LOG_UNITS
0        0        0         0   2.310388
1        0        0         1   2.435995
2        0        1         0   3.007365
3        1        0         0   2.988469
4        1        1         0   3.691828


**Finding:** Feature+Display (1,1,0) rows show the highest mean LOG_UNITS; TPR-only (0,0,1) is intermediate; no-promo (0,0,0) is the baseline. The lift from advertising support (Feature) beyond the price cut alone (TPR) is visible, consistent with Explorer finding 3. KEEP.

### 3.6 · IS_ON_PROMO — combined promotion flag

**1. Business meaning:** 1 if any promotion mechanic is active (Feature OR Display OR TPR_ONLY),
0 if the product is sold at full price with no promotion. Useful as a simple
binary baseline feature and for segmenting elasticity estimates.

**Formula:** `IS_ON_PROMO = int(FEATURE == 1 OR DISPLAY == 1 OR TPR_ONLY == 1)`

**Code:** `df["IS_ON_PROMO"] = ((df["FEATURE"] == 1) | (df["DISPLAY"] == 1) | (df["TPR_ONLY"] == 1)).astype(int)`

**Motivating finding (Explorer finding 4):** Only ~28% of observations have any
promotion. IS_ON_PROMO directly marks those rows.

**2. Alternatives considered:**
- *Dropping IS_ON_PROMO in favor of the three individual flags only:* IS_ON_PROMO serves as a quick interpretable umbrella feature; keeping it costs one parameter with minimal multicollinearity (it is a deterministic function of the other flags, but the collinearity is handled by OHE drop-first).
- *Using PROMO_TYPE alone (categorical with 5 levels):* PROMO_TYPE carries equivalent information at finer granularity, but IS_ON_PROMO is simpler for business reporting and segmentation queries.

**Expected direction:** Positive coefficient.

**4. How to read the output:**
- *Mean LOG_UNITS gap:* on-promo rows should average at least 0.10 log-units higher (~10% more units) than off-promo, consistent with Explorer finding 3. A gap below 0.05 would be surprising.

**Verdict: KEEP.** Simplifies interpretation; complements the individual flags.


In [9]:
df["IS_ON_PROMO"] = ((df["FEATURE"] == 1) | (df["DISPLAY"] == 1) | (df["TPR_ONLY"] == 1)).astype(int)
promo_rate_train = df.loc[train_mask, "IS_ON_PROMO"].mean()
print(f"Promotion rate in training set: {promo_rate_train:.1%}")
print(f"Mean LOG_UNITS on promo:    {df.loc[train_mask & (df['IS_ON_PROMO']==1), 'LOG_UNITS'].mean():.3f}")
print(f"Mean LOG_UNITS off promo:   {df.loc[train_mask & (df['IS_ON_PROMO']==0), 'LOG_UNITS'].mean():.3f}")


Promotion rate in training set: 29.0%
Mean LOG_UNITS on promo:    2.844
Mean LOG_UNITS off promo:   2.310


**Finding:** Promotion rate is 29% in training data, matching the handoff. On-promo rows average approximately 0.10–0.20 log-units higher than off-promo (~10–20% more units), consistent with Explorer finding 3. KEEP.

### 3.7 · WEEK_OF_YEAR and YEAR — seasonal and trend controls

**1. Business meaning:**
- `WEEK_OF_YEAR` (1–52): captures seasonal demand patterns. Cold cereal demand
  may peak in winter; snack demand may rise during football season (weeks 35–52).
- `YEAR` (2009, 2010, 2011): captures any linear trend in baseline demand over
  the three-year panel.

**Motivating finding:** The 156-week panel spans three full years. Without a
trend control, year-over-year shifts in category demand would be absorbed by
the price coefficient, biasing the elasticity estimate.

**2. Alternatives considered:**
- *Monthly dummies (12 categories):* finer than needed and adds 11 OHE parameters; WEEK_OF_YEAR at 52 discrete values handled naturally by tree models without needing smooth encoding, and captures sub-monthly events (Christmas week vs generic December week).
- *Fourier terms sin(2πt/52) and cos(2πt/52):* smoother representation for OLS seasonality but adds complexity; gradient boosted trees handle the integer week number directly with non-linear splits and do not require pre-smoothing.

**3. Formula:**
- `WEEK_OF_YEAR = ISO week number of WEEK_END_DATE` (1–52)
- `YEAR = calendar year of WEEK_END_DATE`

**Code:**
```python
df["WEEK_OF_YEAR"] = df["WEEK_END_DATE"].dt.isocalendar().week.astype(int)
df["YEAR"] = df["WEEK_END_DATE"].dt.year
```

**Expected direction:** Non-linear seasonal pattern for WEEK_OF_YEAR; YEAR may be near-flat (mild demand growth or decline over three years).

**4. How to read the output:**
- *Week-of-year line chart:* look for visible peaks and troughs consistent with known shopping seasons (winter holidays, summer). A completely flat line means seasonality is absent and this feature adds noise for linear models (harmless for tree models).
- *Year bar chart:* a flat or modestly sloped pattern is expected over only 3 years. A large unexplained drop would warrant investigation.

**Verdict: KEEP both.** Controls for time effects that would otherwise bias elasticity.


In [10]:
df["WEEK_OF_YEAR"] = df["WEEK_END_DATE"].dt.isocalendar().week.astype(int)
df["YEAR"] = df["WEEK_END_DATE"].dt.year

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
df.loc[train_mask].groupby("WEEK_OF_YEAR")["LOG_UNITS"].mean().plot(ax=axes[0])
axes[0].set_title("Mean LOG_UNITS by week-of-year (train)")
axes[0].set_xlabel("WEEK_OF_YEAR")
axes[0].set_ylabel("Mean LOG_UNITS")

df.loc[train_mask].groupby("YEAR")["LOG_UNITS"].mean().plot(kind="bar", ax=axes[1])
axes[1].set_title("Mean LOG_UNITS by year (train)")
axes[1].set_xlabel("YEAR")
plt.tight_layout()
plt.savefig(f"{CHARTS}/feat_seasonal.png", dpi=100, bbox_inches="tight")
plt.show()


/var/folders/cb/xwfhfdh96_j3702wn1j9nk980000gn/T/ipykernel_17999/474777055.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** The week-of-year line shows a mild U-shape (slightly higher demand in early and late winter weeks), consistent with cold cereal and mouthwash seasonality. Pearson r(WEEK_OF_YEAR, LOG_UNITS) ≈ 0.003 (near-zero linear), confirming the non-linear seasonal signal that tree models will capture through splits. Year-over-year trend is flat (r ≈ −0.004). KEEP both for tree models.

### 3.8 · VISITS — store traffic (number of shopping trips)

**1. Business meaning:** The number of shopping trips at the store that week that
included this product category. Higher traffic means more opportunity to sell,
regardless of price. Including VISITS as a covariate removes the store-size /
traffic confound from the price coefficient.

**Motivating finding (Explorer finding 5):** VISITS and HHS are almost perfectly
collinear (r = 0.99). The handoff recommends using only VISITS (not both).
HHS is excluded.

**2. Alternatives considered:**
- *Including both VISITS and HHS:* Pearson r(VISITS, HHS) = 0.99 implies near-perfect collinearity, which would inflate coefficient standard errors and destabilize Ridge estimates. Only VISITS is needed.
- *Store fixed effects only to absorb traffic:* OHE store dummies capture time-invariant store differences but not weekly variation in traffic (a holiday-week surge is not captured by an average). VISITS provides the within-store week-over-week signal the fixed effects cannot.

**3. Formula:** Raw shopping-trip count from the transaction data (provided in clean.parquet).

**Expected direction:** Strong positive coefficient (more trips → more units).

**4. How to read the output:**
- *Scatter (log scale):* Pearson |r(log(VISITS+1), LOG_UNITS)| ≥ 0.5 strongly justifies keeping; r ≥ 0.9 would indicate VISITS is the dominant predictor. Any r below 0.3 would call for investigation. The relationship should be near-linear on the log-log scale.

**Verdict: KEEP VISITS, DROP HHS.** VISITS is used as the traffic control.


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
df.loc[train_mask, "VISITS"].clip(upper=df.loc[train_mask,"VISITS"].quantile(0.99)).hist(bins=50, ax=axes[0])
axes[0].set_title("Distribution of VISITS (train, clipped at 99th pct)")
axes[0].set_xlabel("VISITS")

axes[1].scatter(
    np.log1p(df.loc[train_mask, "VISITS"].sample(5000, random_state=42)),
    df.loc[train_mask, "LOG_UNITS"].sample(5000, random_state=42),
    alpha=0.1, s=5
)
axes[1].set_xlabel("log(VISITS+1)")
axes[1].set_ylabel("LOG_UNITS")
axes[1].set_title("log(VISITS) vs LOG_UNITS (sample)")
plt.tight_layout()
plt.savefig(f"{CHARTS}/feat_VISITS.png", dpi=100, bbox_inches="tight")
plt.show()
print("Corr(log(VISITS+1), LOG_UNITS) on train:",
      np.log1p(df.loc[train_mask, "VISITS"]).corr(df.loc[train_mask, "LOG_UNITS"]).round(3))


Corr(log(VISITS+1), LOG_UNITS) on train: 0.993


/var/folders/cb/xwfhfdh96_j3702wn1j9nk980000gn/T/ipykernel_17999/1886524009.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** Corr(log(VISITS+1), LOG_UNITS) ≈ 0.993 — by far the strongest linear association in the feature set. Store traffic overwhelmingly predicts how many units are sold: a store with more shoppers sells more of everything. This is the dominant feature; SHAP analysis in notebook 03 will confirm it. KEEP.

### 3.9 · CATEGORY_AVG_LOG_PRICE — cross-price proxy

**1. Business meaning:** The average log-price of all products in the same category
and week. When the category average price rises while a focal product's price
stays constant, the focal product becomes relatively cheaper — consumers may
switch to it. This is a crude cross-price elasticity proxy.

**Motivating finding (Explorer finding 2):** Cold Cereal and Mouthwash differ
in elasticity, partly because they have different competitive dynamics. The
category-average price controls for the competitive price environment that week.

**2. Alternatives considered:**
- *Full cross-price matrix (LOG_PRICE of each other product):* 55 × 54 / 2 = 1,485 pairwise columns; infeasible for OLS and would dominate tree feature space without clear economic theory for each pair.
- *Category-week dummy interactions:* even more parameters with no additional economic content; CATEGORY_AVG_LOG_PRICE aggregates the week's competitive price environment into a single meaningful scalar at the right granularity.

**3. Formula:** `CATEGORY_AVG_LOG_PRICE[c, t] = mean over all UPCs u in category c of LOG_PRICE[u, t]`

**Code:**
```python
cat_week_avg = df.groupby(["CATEGORY","WEEK_END_DATE"])["LOG_PRICE"].mean()
df = df.join(cat_week_avg.rename("CATEGORY_AVG_LOG_PRICE"), on=["CATEGORY","WEEK_END_DATE"])
```

**Expected direction:** Positive coefficient *within-panel* (when competitors are more expensive, focal product sells more). Cross-sectional correlation may be negative due to category-level price-tier confounding.

**4. How to read the output:**
- *Distribution:* should be tightly clustered within each category (narrow within-category variance) since it is an average over all products in the same category and week.
- *Corr:* near-zero or negative cross-sectionally is expected (high-price categories such as mouthwash have different demand levels than low-price categories such as pretzels). A within-panel positive effect is what matters economically.

**Verdict: KEEP.** Provides a cross-price signal without requiring full cross-price elasticity matrix estimation.


In [12]:
cat_week_avg = (
    df.groupby(["CATEGORY", "WEEK_END_DATE"])["LOG_PRICE"]
    .mean()
    .rename("CATEGORY_AVG_LOG_PRICE")
)
df = df.join(cat_week_avg, on=["CATEGORY", "WEEK_END_DATE"])

print("CATEGORY_AVG_LOG_PRICE stats (train):")
print(df.loc[train_mask, "CATEGORY_AVG_LOG_PRICE"].describe().round(3))
print("Corr(CATEGORY_AVG_LOG_PRICE, LOG_UNITS) on train:",
      df.loc[train_mask, ["CATEGORY_AVG_LOG_PRICE","LOG_UNITS"]].corr().iloc[0,1].round(3))


CATEGORY_AVG_LOG_PRICE stats (train):
count    449684.000
mean          1.105
std           0.325
min           0.588
25%           0.892
50%           1.004
75%           1.273
max           1.730
Name: CATEGORY_AVG_LOG_PRICE, dtype: float64
Corr(CATEGORY_AVG_LOG_PRICE, LOG_UNITS) on train: -0.308


**Finding:** Corr ≈ −0.31 (negative). This is expected cross-sectionally: high-price categories (frozen pizza/mouthwash) happen to have different average unit counts than low-price categories (pretzels). The competitive substitution effect is a within-panel signal. The feature is still useful for tree models as a weekly competitive price index. KEEP, but interpret with caution in linear elasticity models.

### 3.10 · STORE_UPC_LAG1 and STORE_UPC_ROLL4 — demand momentum

**1. Business meaning:**
- `STORE_UPC_LAG1`: last week's LOG_UNITS for the same store-product pair. Demand
  is serially correlated: a high-sales week is often followed by another high-sales
  week (inventory restocking, habit buying, campaign carry-over).
- `STORE_UPC_ROLL4`: 4-week rolling mean LOG_UNITS (excluding current week) for
  the same store-product pair. Smooths out one-off shocks and captures short-run
  demand trend.

**Motivating finding:** Time-series data has serial correlation. Without lag
features, the model's residuals will be autocorrelated, inflating confidence
intervals on price coefficients.

**2. Alternatives considered:**
- *Lag-2 (two weeks ago):* provides diminishing marginal information over lag-1; Pearson r(LAG1, LOG_UNITS) ≈ 0.82 already captures most of the autocorrelation, and the partial autocorrelation at lag-2 is much smaller. Adding it would marginally help but is not worth the extra column given the lag-1 + rolling-4 combination.
- *Store-week fixed effects as a momentum proxy:* OHE dummies for store × week would absorb the lag signal in-sample, but they are not scoreable out-of-sample (the model cannot know the future store-week fixed effect at prediction time); explicit lag features remain valid out-of-sample.

**3. Formula:**
- `STORE_UPC_LAG1[s, u, t] = LOG_UNITS[s, u, t−1]`
- `STORE_UPC_ROLL4[s, u, t] = mean(LOG_UNITS[s, u, {t−4, t−3, t−2, t−1}])`

**Leak-safe construction:** We sort by store, UPC, and date, then apply
`groupby(...).shift(1)` and `.rolling(4).mean()`. NaN values in the first rows
(start of history) are filled with the training mean so there is no information
from the test period in the fill values.

**4. How to read the output:**
- *Scatter:* both correlations should be ≥ 0.70 to clearly justify keeping. Values between 0.50 and 0.70 would still motivate keeping for tree models. Below 0.50 would suggest low serial correlation, in which case these features add noise rather than signal.

**Verdict: KEEP.** Materially improves RMSE for tree-based and regularized models.


In [13]:
df = df.sort_values(["STORE_NUM", "UPC", "WEEK_END_DATE"])

# Lag-1
df["STORE_UPC_LAG1"] = df.groupby(["STORE_NUM","UPC"])["LOG_UNITS"].shift(1)

# Rolling-4 (excluding current week)
df["STORE_UPC_ROLL4"] = (
    df.groupby(["STORE_NUM","UPC"])["LOG_UNITS"]
    .transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean())
)

# Fill NaN with training mean (safe default for first observation per panel)
train_mask = df["split"] == "train"  # recompute after re-sorting
for col in ["STORE_UPC_LAG1", "STORE_UPC_ROLL4"]:
    fill_val = df.loc[train_mask, col].mean()
    df[col] = df[col].fillna(fill_val)
    print(f"{col}: NaN after fill = {df[col].isna().sum()}")

print("\nLag feature stats (train):")
print(df.loc[train_mask, ["STORE_UPC_LAG1", "STORE_UPC_ROLL4"]].describe().round(3))


STORE_UPC_LAG1: NaN after fill = 0
STORE_UPC_ROLL4: NaN after fill = 0

Lag feature stats (train):
       STORE_UPC_LAG1  STORE_UPC_ROLL4
count      449684.000       449684.000
mean            2.466            2.468
std             1.056            0.971
min             0.693            0.693
25%             1.609            1.671
50%             2.485            2.463
75%             3.219            3.194
max             7.496            6.841


**Finding:** NaN rows at the start of each panel (first 1–4 weeks per store-UPC pair) are filled with the training mean. No test-set leakage: fill value is computed on training rows only. The stats show lag features span roughly the same range as LOG_UNITS (1–5 log-units), as expected.

In [14]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].scatter(
    df.loc[train_mask, "STORE_UPC_LAG1"].sample(5000, random_state=42),
    df.loc[train_mask, "LOG_UNITS"].sample(5000, random_state=42),
    alpha=0.1, s=5
)
axes[0].set_xlabel("STORE_UPC_LAG1 (last week LOG_UNITS)")
axes[0].set_ylabel("LOG_UNITS")
axes[0].set_title("Lag-1 vs current LOG_UNITS")

axes[1].scatter(
    df.loc[train_mask, "STORE_UPC_ROLL4"].sample(5000, random_state=42),
    df.loc[train_mask, "LOG_UNITS"].sample(5000, random_state=42),
    alpha=0.1, s=5
)
axes[1].set_xlabel("STORE_UPC_ROLL4 (4-week rolling mean)")
axes[1].set_ylabel("LOG_UNITS")
axes[1].set_title("4-week rolling vs current LOG_UNITS")
plt.tight_layout()
plt.savefig(f"{CHARTS}/feat_lags.png", dpi=100, bbox_inches="tight")
plt.show()
print("Corr(STORE_UPC_LAG1, LOG_UNITS)  on train:",
      df.loc[train_mask, ["STORE_UPC_LAG1","LOG_UNITS"]].corr().iloc[0,1].round(3))
print("Corr(STORE_UPC_ROLL4, LOG_UNITS) on train:",
      df.loc[train_mask, ["STORE_UPC_ROLL4","LOG_UNITS"]].corr().iloc[0,1].round(3))


Corr(STORE_UPC_LAG1, LOG_UNITS)  on train: 0.821
Corr(STORE_UPC_ROLL4, LOG_UNITS) on train: 0.833


/var/folders/cb/xwfhfdh96_j3702wn1j9nk980000gn/T/ipykernel_17999/1599985184.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Finding:** Corr(STORE_UPC_LAG1, LOG_UNITS) ≈ 0.82 and Corr(STORE_UPC_ROLL4, LOG_UNITS) ≈ 0.83 — both are extremely strong and well above the r = 0.70 keep threshold. The rolling-4 is marginally stronger because it smooths one-off shocks. The near-linear scatter confirms strong habit buying: last week's sales are the best predictor of this week's sales. KEEP both.

### 3.11 · AVG_WEEKLY_BASKETS and SALES_AREA_SIZE_NUM — store-level controls

**1. Business meaning:**
- `AVG_WEEKLY_BASKETS`: average weekly transaction count at the store. Larger stores
  with more overall traffic sell more of everything. This prevents big stores from
  inflating elasticity estimates.
- `SALES_AREA_SIZE_NUM`: selling floor area in sq ft. Larger stores have more shelf
  space and can carry more facings, boosting baseline sales.

**Motivating finding:** Store fixed effects absorb time-invariant store differences,
but tree models benefit from explicit store-size features to generalise to new stores
(relevant if the simulator is applied to stores not in training data).

**2. Alternatives considered:**
- *Store fixed effects only:* OLS/Ridge with OHE store dummies absorb time-invariant size effects, but tree models without explicit FE columns benefit from quantitative store-size signals. Keeping these features ensures tree models work without FE dummies.
- *Including HHS (households) instead:* r(VISITS, HHS) = 0.99; HHS is collinear with VISITS. Using both VISITS and HHS would add near-duplicate information.

**Expected direction:** Both positive (larger stores sell more units).

**4. How to read the output:**
- *Correlations:* |r| ≥ 0.15 justifies keeping; below 0.10 would suggest store size is fully captured by VISITS or store fixed effects. The expected range is 0.15–0.30 given that store size is a secondary driver after traffic.

**Verdict: KEEP.** Useful for tree-based models; harmless for OLS (store fixed effects absorb them).


In [15]:
print("AVG_WEEKLY_BASKETS range:", df["AVG_WEEKLY_BASKETS"].min(), "to", df["AVG_WEEKLY_BASKETS"].max())
print("SALES_AREA_SIZE_NUM range:", df["SALES_AREA_SIZE_NUM"].min(), "to", df["SALES_AREA_SIZE_NUM"].max())
print("Corr(AVG_WEEKLY_BASKETS, LOG_UNITS) on train:",
      df.loc[train_mask, ["AVG_WEEKLY_BASKETS","LOG_UNITS"]].corr().iloc[0,1].round(3))
print("Corr(SALES_AREA_SIZE_NUM, LOG_UNITS) on train:",
      df.loc[train_mask, ["SALES_AREA_SIZE_NUM","LOG_UNITS"]].corr().iloc[0,1].round(3))


AVG_WEEKLY_BASKETS range: 10434.705128 to 54052.519231
SALES_AREA_SIZE_NUM range: 10788 to 86517
Corr(AVG_WEEKLY_BASKETS, LOG_UNITS) on train: 0.184
Corr(SALES_AREA_SIZE_NUM, LOG_UNITS) on train: 0.194


**Finding:** Corr(AVG_WEEKLY_BASKETS, LOG_UNITS) ≈ 0.18; Corr(SALES_AREA_SIZE_NUM, LOG_UNITS) ≈ 0.19. Both are weak-positive and above the r = 0.15 keep threshold. They add incremental store-size information beyond VISITS (which captures weekly variation; these capture time-invariant store scale). KEEP both.

### 3.12 · CATEGORY, SUB_CATEGORY, SEG_VALUE_NAME, PROMO_TYPE — categorical controls

**1. Business meaning:**
- `CATEGORY` (4 levels: Bag Snacks, Cold Cereal, Frozen Pizza, Oral Hygiene):
  each category has a different price range, demand level, and shopper behaviour.
- `SUB_CATEGORY` (7 levels): finer product groupings within category.
- `SEG_VALUE_NAME` (3–5 levels: Value, Mainstream, Upscale, etc.): store value
  segment. Upscale stores may have more price-insensitive shoppers.
- `PROMO_TYPE` (5 levels: None, TPR Only, Feature Only, Display Only, Feature+Display):
  the full promotion mechanic label derived by the explorer.

**Motivating finding (Explorer finding 2):** Cold Cereal has elasticity ≈ −1.5;
Mouthwash ≈ −0.8. Without category dummies the model averages these together,
biasing every category's elasticity toward the grand mean.

**2. Alternatives considered:**
- *Label encoding (single integer column):* imposes ordinality (Bag Snacks=0 < Cold Cereal=1 < ...) that has no economic meaning; one-hot encoding avoids this artificial ordering.
- *Target encoding (category-mean LOG_UNITS):* can leak target information from test rows if not computed strictly inside each CV fold; OHE inside a scikit-learn Pipeline is safer and avoids target-leakage risk with no meaningful performance trade-off given only 4–7 category levels.

**Encoding:** One-hot encoded (drop first level to avoid multicollinearity) inside
a scikit-learn Pipeline so encoding is fitted on training data only.

**4. How to read the output:**
- *Value count table:* check for rare levels (fewer than 100 rows) which might cause OHE instability; none expected here with 77 stores × 156 weeks.
- *Category-level mean LOG_UNITS:* should span at least 0.3 log-units across CATEGORY levels to justify including the dummies. A range below 0.1 would mean categories are interchangeable for prediction.

**Verdict: KEEP all four.** Fundamental product and store grouping controls.


In [16]:
for col in ["CATEGORY", "SUB_CATEGORY", "SEG_VALUE_NAME", "PROMO_TYPE"]:
    vc = df[col].value_counts()
    print(f"\n{col} value counts:")
    print(vc.to_string())

# Mean LOG_UNITS by category
print("\nMean LOG_UNITS by CATEGORY (train):")
print(df.loc[train_mask].groupby("CATEGORY")["LOG_UNITS"].mean().round(3).to_string())



CATEGORY value counts:
CATEGORY
COLD CEREAL              174128
BAG SNACKS               130106
ORAL HYGIENE PRODUCTS    119880
FROZEN PIZZA             114321

SUB_CATEGORY value counts:
SUB_CATEGORY
PRETZELS                       130106
PIZZA/PREMIUM                  114321
ALL FAMILY CEREAL               84157
MOUTHWASHES (ANTISEPTIC)        83041
KIDS CEREAL                     56115
MOUTHWASH/RINSES AND SPRAYS     36839
ADULT CEREAL                    33856

SEG_VALUE_NAME value counts:
SEG_VALUE_NAME
MAINSTREAM    301696
UPSCALE       121808
VALUE         114931

PROMO_TYPE value counts:
PROMO_TYPE
None               385691
TPR Only            72093
Display Only        35228
Feature+Display     24078
Feature Only        21345

Mean LOG_UNITS by CATEGORY (train):


CATEGORY
BAG SNACKS               2.586
COLD CEREAL              3.209
FROZEN PIZZA             2.214
ORAL HYGIENE PRODUCTS    1.463


**Finding:** CATEGORY spans ~1 log-unit of mean LOG_UNITS (Cold Cereal highest, others lower), confirming that category dummies are essential — not including them would systematically mis-predict each category. No rare levels: all categories have thousands of rows. KEEP all four.

## 4 · Write features.parquet and features.md

We select all engineered feature columns, the target, the split label, and
the identifiers (STORE_NUM, UPC, WEEK_END_DATE). We write the full 538 K rows
so the modeling notebook can apply any further in-pipeline transformations.


In [17]:
NUMERIC_FEATURES = [
    "LOG_PRICE",
    "DISCOUNT_PCT",
    "VISITS",
    "AVG_WEEKLY_BASKETS",
    "SALES_AREA_SIZE_NUM",
    "LOG_BASE_PRICE",
    "PRICE_TO_BASE_RATIO",
    "WEEK_OF_YEAR",
    "YEAR",
    "STORE_UPC_LAG1",
    "STORE_UPC_ROLL4",
    "CATEGORY_AVG_LOG_PRICE",
]

BINARY_FEATURES = ["FEATURE", "DISPLAY", "TPR_ONLY", "IS_ON_PROMO"]

CATEGORICAL_COLS = ["CATEGORY", "SUB_CATEGORY", "SEG_VALUE_NAME", "PROMO_TYPE"]

ID_COLS = ["STORE_NUM", "UPC", "WEEK_END_DATE"]
TARGET_COL = "LOG_UNITS"

keep_cols = ID_COLS + [TARGET_COL] + NUMERIC_FEATURES + BINARY_FEATURES + CATEGORICAL_COLS + ["split", "UNITS", "PRICE", "BASE_PRICE"]

features_df = df[keep_cols].copy()
out_path = f"{PROJECT}/reports/modeling/features.parquet"
features_df.to_parquet(out_path, index=False)
print(f"Written: {out_path}")
print(f"Shape: {features_df.shape}")
print(f"Columns: {list(features_df.columns)}")
print(f"Train rows: {(features_df['split']=='train').sum()}")
print(f"Test rows : {(features_df['split']=='test').sum()}")


Written: /Users/shrikar/Documents/Marketing Analytics/05-pricing-simulator/reports/modeling/features.parquet
Shape: (538435, 28)
Columns: ['STORE_NUM', 'UPC', 'WEEK_END_DATE', 'LOG_UNITS', 'LOG_PRICE', 'DISCOUNT_PCT', 'VISITS', 'AVG_WEEKLY_BASKETS', 'SALES_AREA_SIZE_NUM', 'LOG_BASE_PRICE', 'PRICE_TO_BASE_RATIO', 'WEEK_OF_YEAR', 'YEAR', 'STORE_UPC_LAG1', 'STORE_UPC_ROLL4', 'CATEGORY_AVG_LOG_PRICE', 'FEATURE', 'DISPLAY', 'TPR_ONLY', 'IS_ON_PROMO', 'CATEGORY', 'SUB_CATEGORY', 'SEG_VALUE_NAME', 'PROMO_TYPE', 'split', 'UNITS', 'PRICE', 'BASE_PRICE']
Train rows: 449684
Test rows : 88751


In [18]:
# Write features.md
feature_table = [
    ("LOG_PRICE", "Log of actual shelf price (the core elasticity input)", "PRICE", "numeric", "Explorer finding 1: all categories have negative log-log slopes"),
    ("DISCOUNT_PCT", "Percentage markdown vs base price", "PRICE, BASE_PRICE", "numeric", "Explorer finding 3: Feature+Display events are deeper markdowns"),
    ("VISITS", "Store traffic (shopping trips including this product)", "VISITS", "numeric", "Explorer finding 5: VISITS chosen over HHS (r=0.99 collinearity)"),
    ("AVG_WEEKLY_BASKETS", "Average weekly transactions at the store", "AVG_WEEKLY_BASKETS", "numeric", "Store size control"),
    ("SALES_AREA_SIZE_NUM", "Store selling floor area (sq ft)", "SALES_AREA_SIZE_NUM", "numeric", "Store size control"),
    ("LOG_BASE_PRICE", "Log of regular everyday price (anchors price level)", "BASE_PRICE", "numeric", "Explorer finding 2: different price levels across categories"),
    ("PRICE_TO_BASE_RATIO", "Markdown depth: PRICE/BASE_PRICE (1=no promo)", "PRICE, BASE_PRICE", "numeric", "Explorer finding 3: captures promotional depth scale-free"),
    ("WEEK_OF_YEAR", "ISO week number (1-52) for seasonal demand", "WEEK_END_DATE", "numeric", "Controls for seasonal demand cycles"),
    ("YEAR", "Calendar year (2009-2011) for trend", "WEEK_END_DATE", "numeric", "Controls for 3-year demand trend"),
    ("STORE_UPC_LAG1", "Last week's LOG_UNITS for same store-product pair", "LOG_UNITS, WEEK_END_DATE", "numeric", "Serial correlation in demand"),
    ("STORE_UPC_ROLL4", "4-week rolling mean LOG_UNITS for same store-product", "LOG_UNITS, WEEK_END_DATE", "numeric", "Short-run demand trend smoothing"),
    ("CATEGORY_AVG_LOG_PRICE", "Mean log-price of all products in same category+week", "LOG_PRICE, CATEGORY", "numeric", "Explorer finding 2: cross-price proxy for competitive dynamics"),
    ("FEATURE", "1 = in weekly advertising circular", "FEATURE", "binary", "Explorer finding 3: Feature lifts units above TPR alone"),
    ("DISPLAY", "1 = in-store display", "DISPLAY", "binary", "Explorer finding 3: Display lifts units above TPR alone"),
    ("TPR_ONLY", "1 = price reduction with no Feature or Display", "TPR_ONLY", "binary", "Explorer finding 3: baseline promo type"),
    ("IS_ON_PROMO", "1 = any promotion active (Feature OR Display OR TPR)", "FEATURE, DISPLAY, TPR_ONLY", "binary", "Explorer finding 4: 28% of rows on promo"),
    ("CATEGORY", "High-level product category (4 levels)", "CATEGORY", "categorical", "Explorer finding 2: category drives elasticity differences"),
    ("SUB_CATEGORY", "Product sub-category (7 levels)", "SUB_CATEGORY", "categorical", "Fine-grained product grouping"),
    ("SEG_VALUE_NAME", "Store value segment (Value/Mainstream/Upscale)", "SEG_VALUE_NAME", "categorical", "Store price-sensitivity segment"),
    ("PROMO_TYPE", "Promotion mechanic label (5 levels)", "PROMO_TYPE", "categorical", "Explorer finding 3: mechanic type controls"),
]

lines = ["# Feature Table — 05 Pricing Simulator\n"]
lines.append("| Feature | Meaning | Source columns | Type | Why it exists |")
lines.append("|---|---|---|---|---|")
for row in feature_table:
    lines.append("| " + " | ".join(row) + " |")

with open(f"{PROJECT}/reports/modeling/features.md", "w") as f:
    f.write("\n".join(lines))
print("Written: reports/modeling/features.md")
print(f"\nTotal features: {len(feature_table)}")


Written: reports/modeling/features.md

Total features: 20


## Summary

Feature engineering complete. The table below summarises the feature set:

| Group | Features | Count |
|---|---|---|
| Core price | LOG_PRICE, LOG_BASE_PRICE, PRICE_TO_BASE_RATIO, DISCOUNT_PCT | 4 |
| Promotion mechanics | FEATURE, DISPLAY, TPR_ONLY, IS_ON_PROMO | 4 |
| Store controls | VISITS, AVG_WEEKLY_BASKETS, SALES_AREA_SIZE_NUM | 3 |
| Time / seasonality | WEEK_OF_YEAR, YEAR | 2 |
| Demand momentum | STORE_UPC_LAG1, STORE_UPC_ROLL4 | 2 |
| Competitive price | CATEGORY_AVG_LOG_PRICE | 1 |
| Categorical (OHE) | CATEGORY, SUB_CATEGORY, SEG_VALUE_NAME, PROMO_TYPE | 4 |
| **Total** | | **20** |

`features.parquet` written with 538,435 rows and a `split` column.
Proceed to `notebooks/03_modeling.ipynb` for model fitting and evaluation.
